# Sets, fields, and distributions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/03_sets_fields_distributions.ipynb)

Minilink uses three distinct mathematical objects for regions, scalar functions, and uncertainty:
- **`Set`** and **`InputSet`**: deterministic feasibility regions $z \in Z(t)$ and $u \in U(x, t)$ defined by nonnegative margins $\mathrm{margin} \ge 0$;
- **`Field`**: scalar functions $V(x, u, t) \in \mathbb{R}$ (such as energy or Lyapunov candidates) and their gradients $\nabla_x V$;
- **`Distribution`**: probability laws $z \sim p$ with `.sample(key, n)`, `.mean`, `.cov`, and optional `.support`.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `Set`: allowable state and boundary regions

Every `Set` represents $z \in Z(t) \iff \mathrm{margin}(z, t) \ge 0$ componentwise. Positive margin is inside, zero is on the boundary, and negative is outside:

| `Set` Method / Property | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `S.dim` | — | `int` | Vector dimension $n$ (`BoxSet`, `BallSet`, `SingletonSet`, `EllipsoidSet`) |
| `S.params` | — | `dict[str, Any]` | Defining numbers (`{"lower", "upper"}`, `{"center", "radius"}`, `{"point"}`, ...), edited in place: `S.params["upper"] = ...` |
| `S.margin(z, t=0.0, params=None)` | `z: array (n,)`, `t: float = 0.0`, `params: dict \| None = None` | `array (k,)` | Feasibility margins ($k = 2n$ for `BoxSet`, $1$ for `BallSet`/`EllipsoidSet`, $n$ for `SingletonSet`) |
| `S.contains(z, t=0.0, params=None)` | `z: array (n,)`, `t: float = 0.0`, `params: dict \| None = None` | `bool` (or JAX bool) | `True` iff all components of `S.margin(z, t, params)` are $\ge 0$ |
| `S.sample(key, n=None, params=None)` | `key: int \| Generator \| jax.Array`, `n: int \| None = None`, `params=None` | `array (n_dim,)` if `n is None`, else `(n, n_dim)` | Draw uniformly from the set (`BoxSet`, `EllipsoidSet`) |
| `S.bounding_box()` | — | `BoxSet \| None` | Tightest axis-aligned `BoxSet` containing `S`, or `None` if unbounded |
| `S.as_cost(*, weight=1.0, shaping=None)` | `weight: float = 1.0`, `shaping: Callable \| None = None` | `CostFunction` | Running penalty $g(x) = w \cdot \mathrm{shaping}(\min_i m_i(x))$ (default: quadratic hinge outside `S`) |
| `S1 & S2` (`IntersectionSet.of(S1, S2)`) | `other: Set` | `IntersectionSet` | Intersection $z \in S_1 \cap S_2$; concatenates member margin vectors |

### `BoxSet`: axis-aligned bounds $\text{lower} \le z \le \text{upper}$

For `BoxSet(lower, upper)`, `margin(z)` returns the $2n$-vector $[z - \text{lower};\; \text{upper} - z]$:

In [ ]:
import numpy as np

from minilink import BallSet, BoxSet, Pendulum, SingletonSet

X = BoxSet([-np.pi, -4.0], [np.pi, 4.0])
z_out = np.array([1.0, 5.0])

X.dim, X.params

In [ ]:
X.margin(z_out)

In [ ]:
X.contains(z_out)

In [ ]:
z_in = np.array([1.0, 2.0])
X.contains(z_in)

`X.sample(key, n=None)` returns a single point of shape `(dim,)` when `n` is omitted, or a matrix of shape `(n, dim)` when `n` is given:

In [ ]:
single_point = X.sample(0)
three_points = X.sample(0, n=3)
single_point.shape, three_points.shape, three_points

In [ ]:
X.bounding_box()

A system exposes its declared state box on `plant.state.box`, and its input box on `plant.input_box`:

In [ ]:
plant = Pendulum()
plant.state.box, plant.input_box

### `BallSet`, `SingletonSet`, `CallableSet`, and `IntersectionSet` (`&`)

- **`BallSet(center, radius)`**: $\|z - c\|_2 \le r$; `margin(z)` has shape `(1,)` with value $r - \|z - c\|_2$.
- **`SingletonSet(point)`**: exact equality $z = p$; `residual(z)` returns $z - p$ of shape `(n,)` (used as equality constraints $h(z) = 0$ by optimizers), while `margin(z)` returns $-|z - p| \le 0$.
- **`CallableSet(margin_fn=None, contains_fn=None, params=None)`**: custom set defined by a user function `margin_fn(z, t, params) -> array`.
- **`A & B` (`IntersectionSet`)**: concatenates the margin vectors of its members and intersects their bounding boxes.

In [ ]:
goal_ball = BallSet(center=[np.pi, 0.0], radius=0.2)
goal_ball.margin(np.array([np.pi + 0.05, 0.0])), goal_ball.bounding_box()

In [ ]:
goal_point = SingletonSet([np.pi, 0.0])
z_test = np.array([3.0, 0.1])
goal_point.residual(z_test), goal_point.margin(z_test), goal_point.contains(np.array([np.pi, 0.0]))

In [ ]:
from minilink.core import Set


class HalfspaceSet(Set):
    def __init__(self, limit=1.0):
        self.params = {"limit": float(limit)}

    def margin(self, z, t=0.0, params=None):
        params = self.params if params is None else params
        limit = params["limit"]
        m = np.array([limit - z[0]])
        return m


halfspace = HalfspaceSet(limit=1.0)
halfspace.margin(np.array([0.4, 0.0])), halfspace.contains(np.array([1.5, 0.0]))

In [ ]:
safe_goal = X & goal_ball
safe_goal.margin(np.array([np.pi, 0.0])).shape, safe_goal.bounding_box()

## 2. `InputSet` and `BoxInputSet`: admissible controls $u \in U(x, t)$

`InputSet` is the control counterpart of `Set`: its `margin(u, x=None, t=0.0, params=None)` and `contains(u, x=None, t=0.0, params=None)` take the control `u` first and optionally the current state `x`.

| `InputSet` / `BoxInputSet` API | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `BoxInputSet(BoxSet(lower, upper))` | `lower: (m,)`, `upper: (m,)` | `BoxInputSet` | State-independent input box $\text{lower} \le u \le \text{upper}$; `PlanningProblem(U=...)` also takes the `BoxSet` itself |
| `sys.input_box` | — | `BoxInputSet` | The input ports' bounds, stacked in port order |
| `U.margin(u, x=None, t=0.0, params=None)` | `u: (m,)`, `x: (n,) \| None = None`, `t: float = 0.0`, `params=None` | `array (2m,)` | Input feasibility margins $[u - \text{lower};\; \text{upper} - u]$ |
| `U.contains(u, x=None, t=0.0, params=None)` | `u: (m,)`, `x: (n,) \| None = None`, `t: float = 0.0`, `params=None` | `bool` | `True` iff $u$ is admissible at $(x, t)$ |
| `U.sample(key, n=None, x=None, t=0.0, params=None)` | `key`, `n: int \| None = None`, `x=None`, `t=0.0`, `params=None` | `array (m,)` or `(n, m)` | Sample admissible inputs uniformly from the box |
| `U.bounding_box()` | — | `BoxSet` | Underlying `BoxSet` of input bounds |

In [ ]:
from minilink import BoxInputSet

U = BoxInputSet(BoxSet([-5.0], [5.0]))
U.margin(np.array([3.0])), U.contains(np.array([6.0]))

In [ ]:
U.sample(0, n=3), U.bounding_box()

## 3. `Field`: scalar functions $V(x, u, t)$ and conversions to sets and costs

A `Field` evaluates a scalar quantity $V(x, u, t) \in \mathbb{R}$ and its state gradient $\nabla_x V(x, u, t) \in \mathbb{R}^n$:
- **`QuadraticField(center, P, t=None)`**: $V(x, t) = (x - c)^\top P(t) (x - c)$, with $\nabla_x V(x, t) = 2 P(t)(x - c)$.
- **`MarginField(set)`**: $V(x, t) = \min_i m_i(x, t)$, the signed least margin of a `Set`.
- **`GridField(levels, values, t=None, *, method="linear", outside=None)`** (`minilink.core`, and the root): a table on a regular grid read multilinearly in the cell holding $x$ (`"nearest"`, or `"cubic"` and `"quintic"` on NumPy), a time axis read linearly in $t$; outside the grid the edge is held, or $V = $ `outside`. Value iteration's `cost_to_go` is one.

| `Field` Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `V(x, u=None, t=0.0, params=None)` | as `value` | `float` (scalar) | The field called: `V(x)`, time by keyword `V(x, t=t)` (the second argument is `u`) |
| `V.value(x, u=None, t=0.0, params=None)` | `x: (n,)`, `u: (m,) \| None = None`, `t: float = 0.0`, `params=None` | `float` (scalar) | Scalar field value $V(x, u, t)$ |
| `V.gradient(x, u=None, t=0.0, params=None)` | `x: (n,)`, `u: (m,) \| None = None`, `t: float = 0.0`, `params=None` | `array (n,)` | State gradient $\nabla_x V(x, u, t)$ |
| `V.as_set(lower=0.0, upper=None)` | `lower: float \| None = 0.0`, `upper: float \| None = None` | `EllipsoidSet \| FieldSet` | Level-set region $\{x : \text{lower} \le V(x, t) \le \text{upper}\}$ |
| `V.as_input_set(lower=0.0, upper=None)` | `lower: float \| None = 0.0`, `upper: float \| None = None` | `FieldInputSet` | State-dependent input set $\{u : \text{lower} \le V(x, u, t) \le \text{upper}\}$ |
| `V.as_cost(*, weight=1.0, shaping=None)` | `weight: float = 1.0`, `shaping: Callable \| None = None` | `FieldCost (CostFunction)` | Running cost $g(x, u, t) = w \cdot \mathrm{shaping}(V(x, u, t))$ |
| `V.as_terminal_cost(*, weight=1.0, shaping=None)` | `weight: float = 1.0`, `shaping: Callable \| None = None` | `FieldTerminalCost (CostFunction)` | Terminal cost $h(x, t) = w \cdot \mathrm{shaping}(V(x, t))$ |

In [ ]:
from minilink.core import MarginField, QuadraticField

V = QuadraticField(np.zeros(2), np.diag([2.0, 0.5]))
x_query = np.array([1.0, 2.0])

V.value(x_query), V.gradient(x_query)

In [ ]:
E = V.as_set(lower=None, upper=1.0)
running_cost = V.as_cost(weight=2.0)
terminal_cost = V.as_terminal_cost(weight=5.0)
wall_penalty = X.as_cost(weight=10.0)

(
    E,
    E.contains(x_query),
    running_cost.g(x_query, np.zeros(1)),
    terminal_cost.h(x_query),
    wall_penalty.g(z_out, np.zeros(1)),
)

In [ ]:
margin_field = MarginField(X)
margin_field.value(z_in), margin_field.gradient(z_in)

Visualizing an intersection of an ellipsoid level-set `E = V.as_set(lower=None, upper=1.0)` and a `BallSet` inside `X`:

In [ ]:
import matplotlib.pyplot as plt

points = X.sample(0, 2000)
ball = BallSet([0.5, 0.0], 1.2)
E_ball = E & ball
inside = np.array([E_ball.contains(point) for point in points])

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(*points[~inside].T, s=4, c="lightgray", label="X only")
ax.scatter(*points[inside].T, s=4, c="tab:blue", label="E & ball")
ax.set_xlabel("theta")
ax.set_ylabel("dtheta")
ax.legend()
plt.show()

## 4. `Distribution`: probability laws over vectors

A `Distribution` represents a random vector $z \sim p$. Unlike a `Set` (which answers membership and margin), a `Distribution` answers draws (`sample`) and moments (`mean`, `cov`), plus its bounded `support` when one exists:

| `Distribution` Method / Property | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `law.dim` | — | `int` | Dimension $d$ of a drawn vector |
| `law.mean` | — | `ndarray (d,)` | Mean vector $\mathbb{E}[z]$ |
| `law.cov` | — | `ndarray (d, d)` | Covariance matrix $\mathrm{Cov}(z)$ |
| `law.support` | — | `Set \| None` | Bounded support set (`BoxSet` for `Uniform`, `None` for `Gaussian` and `Particles`) |
| `law.params` | — | `dict[str, ndarray]` | Defining parameters (`{"mean", "std"}`, `{"lower", "upper"}`, `{"points"}`), edited in place: `law.params["mean"] = ...` |
| `law.sample(key, n=None, params=None)` | `key: int \| Generator \| jax.Array \| None`, `n: int \| None = None`, `params=None` | `array (d,)` if `n is None`, else `(n, d)` | Draw 1 or `n` samples; `key=None` returns the mean at `params` |


In [ ]:
from minilink import Gaussian, Uniform
from minilink.core import Particles

start_law = Gaussian(mean=[0.0, 0.0], std=[0.1, 0.5])
start_law.dim, start_law.mean, start_law.cov, start_law.support

In [ ]:
# Single draw (n=None) vs 3 draws (n=3) vs deterministic mean draw (key=None)
start_law.sample(0), start_law.sample(0, n=3), start_law.sample(None)

In [ ]:
mass_law = Uniform(lower=[0.8], upper=[1.2])
mass_law.mean, mass_law.cov, mass_law.support

In [ ]:
cloud = Particles(np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 0.0]]))
cloud.dim, cloud.mean, cloud.cov, cloud.sample(0, n=4)

## Summary

| Abstraction | Concrete classes | Main methods & outputs | Role in `PlanningProblem` |
| --- | --- | --- | --- |
| `Set` | `BoxSet`, `BallSet`, `SingletonSet`, `EllipsoidSet`, `FieldSet`, `CallableSet`, `IntersectionSet` | `margin(z, t) -> (k,)`, `contains(z, t) -> bool`, `sample(key, n) -> (n, d)`, `bounding_box() -> BoxSet`, `as_cost() -> CostFunction` | Hard state sets `X`, `X0`, `Xf` |
| `InputSet` | `BoxInputSet`, `FieldInputSet` | `margin(u, x, t) -> (k,)`, `contains(u, x, t) -> bool`, `sample(key, n) -> (n, m)`, `bounding_box() -> BoxSet` | Hard control set `U` |
| `Field` | `QuadraticField`, `GridField`, `MarginField` | `value(x, u, t) -> float`, `gradient(x, u, t) -> (n,)`, `as_set()`, `as_cost()`, `as_terminal_cost()` | Level sets and shaped penalties |
| `Distribution` | `Gaussian`, `Uniform`, `Particles` | `sample(key, n) -> (n, d)`, `mean -> (d,)`, `cov -> (d, d)`, `support -> Set \| None` | Initial law `x0` and parameter laws `params` |

Next, [`04_costs`](04_costs.ipynb): running costs, terminal costs, stage costs, and cost algebra.